# Task 04/05 -- Joint-group selection & triplet coupling (`docs/open_questions.md` §3.2, §2.2)

Fits `correlated_variables.md` §4's joint-DMD-over-independently-Tucker-
compressed-coefficients construction for the wind-sea generation group
(`u_wind,v_wind,u_wsea,v_wsea,T` from `was`/`wsh`/`wsd`/`wsp`) and inspects
the fitted operator's off-diagonal blocks (§5's diagnostic) -- both for
group-selection evidence (Task 04) and for whether it restores the
within-triplet coupling independent Gaussianization discards (Task 05).
See `tasks/04-joint-group-selection.md` and `tasks/05-triplet-coupling-gaussianization.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 12, 1, tzinfo=timezone.utc)

from train.directional import ComingFromToUV

X_raw = load_training_window(DATA_DIR, T_START, T_END, ["was", "wad_come_from", "wsh", "wsd", "wsp"])
X_uv = ComingFromToUV(
    pairs=[
        ("was", "wad_come_from", "u_wind", "v_wind"),
        ("wsh", "wsd", "u_wsea", "v_wsea"),
    ]
).fit_transform(X_raw)

T_period = np.log(X_raw["wsp"])  # variable_taxonomy.md §3.4 log-period convention

def to_lat_lon_time(da):
    return da.transpose("wave_lat", "wave_lon", "time").values

members = {
    "u_wind": to_lat_lon_time(X_uv["u_wind"]),
    "v_wind": to_lat_lon_time(X_uv["v_wind"]),
    "u_wsea": to_lat_lon_time(X_uv["u_wsea"]),
    "v_wsea": to_lat_lon_time(X_uv["v_wsea"]),
    "T_wsp": to_lat_lon_time(T_period),
}
ocean_mask = np.ones(members["u_wsea"].shape[:2], dtype=bool)
for arr in members.values():
    ocean_mask &= ~np.isnan(arr).any(axis=-1)
print(f"{ocean_mask.mean():.1%} shared ocean")


## Independent per-member Tucker fits (`tucker_decomposition.md` §4.1 -- unchanged)

In [ ]:
R_LAT, R_LON = 8, 16
cores = {}
for name, arr in members.items():
    filled = np.where(ocean_mask[:, :, None], arr, 0.0)
    mask3 = np.broadcast_to(ocean_mask[:, :, None], filled.shape).astype(np.float64)
    core, _ = partial_tucker(
        filled, modes=[0, 1, 2], rank=[R_LAT, R_LON, filled.shape[-1]],
        init="random", tol=1e-5, n_iter_max=2000, random_state=SEED, mask=mask3,
    )
    cores[name] = core  # (R_LAT, R_LON, T)


## Joint state + joint DMD across the group (`correlated_variables.md` §4.2)

In [ ]:
from train.dmd import dmd_fit

T = next(iter(cores.values())).shape[-1]
x_t = np.concatenate([c.reshape(-1, T) for c in cores.values()], axis=0)  # (n_V, T)
dt = 3.0 * 3600.0

joint_core = x_t.T[:, None, :]  # dmd_fit wants (T, r_a, r_b); fold everything into r_b
dmd_joint = dmd_fit(joint_core, r_dmd=min(20, x_t.shape[0] - 1), dt=dt, rng=rng)


## Block-structure diagnostic (`correlated_variables.md` §5)

Compare against fitting `train.dmd.dmd_fit` on each member's own core
independently (no concatenation) -- non-negligible cross-member structure
in the joint fit that an independent fit cannot represent at all is the
literal, checkable evidence §5 asks for. Record group-size/off-diagonal-
strength findings in `tasks/04-joint-group-selection.md` §4, and whether
the wsh/wsd/wsp within-triplet block specifically shows non-negligible
coupling in `tasks/05-triplet-coupling-gaussianization.md` §4.

In [ ]:
# TODO (fill in once run against real data): partition dmd_joint.Phi's rows
# by member (index ranges into x_t's concatenation order) and compare
# reconstructed cross-member prediction error against each member's own
# independent dmd_fit.
